In [ ]:
import boto3
import pandas as pd
from decimal import Decimal

# Best practice: convertir Decimales de DynamoDB a floats de Python
class DecimalEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, Decimal):
            return float(obj)
        return json.JSONEncoder.default(self, obj)

# credenciales de AWS deben estar configuradas localmente
dynamodb = boto3.resource('dynamodb', region_name='us-east-2')
table = dynamodb.Table('TesisTwitterData')

# scan para obtener todos los items de la tabla
response = table.scan()
data = response['Items']

# DynamoDB hace paginacion en tablas muy grandes. Este loop asegura obtener todas las páginas.
while 'LastEvaluatedKey' in response:
    response = table.scan(ExclusiveStartKey=response['LastEvaluatedKey'])
    data.extend(response['Items'])

# Guardar datos en dtaframe de pandas
df = pd.DataFrame(data)

# El sentimiento tambien tiene que ser float
# El json.loads(json.dumps(...)) me permite usar DecimalEncoder
df['sentiment_score'] = df['sentiment_score'].apply(lambda x: json.loads(json.dumps(x, cls=DecimalEncoder)))
df['sentiment_score'] = pd.to_numeric(df['sentiment_score'])


print(f"Se han cargado {len(df)} tuits en el DataFrame.")
print("Primeras 5 filas de datos:")
print(df.head())
print("\nColumnas disponibles:")
print(df.columns)

In [ ]:
# Agrupar por medio y por político
proximity_analysis = df.groupby(['autor_medio', 'politico_mencionado']).agg(
    volumen_menciones=('tweet_id', 'count'),
    sentimiento_promedio=('sentiment_score', 'mean'),
    impacto_promedio_likes=('like_count', 'mean')
).reset_index()

print("Análisis de Proximidad:")
print(proximity_analysis)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Tabla pivote para la heatmap de sentimiento
heatmap_data = proximity_analysis.pivot_table(
    index='autor_medio', 
    columns='politico_mencionado', 
    values='sentimiento_promedio'
)

# Crear la visualización
plt.figure(figsize=(10, 7))
sns.heatmap(heatmap_data, annot=True, cmap="coolwarm", fmt=".2f")
plt.title('Sentimiento Promedio de Medios hacia Políticos')
plt.xlabel('Político Mencionado')
plt.ylabel('Medio de Comunicación')
plt.show()